In [1]:
# import libraries
import xpress as xp
import numpy as np

In [2]:
#Empirical parameters (2,4,6,8,10leg)
k001 = [3.66E-06, 1.84E-06, 1.15E-06, 7.65E-07, 5.41E-07]
kact = np.zeros((5, 11))
kact[0,:] = np.asarray((3.66E-06, 5.99E-06, 9.19E-06, 1.34E-05, 1.89E-05, 2.56E-05, 3.38E-05, 4.36E-05, 5.52E-05, 6.86E-05, 8.41E-05))
kact[1,:] = np.asarray((1.84E-06, 3.00E-06, 4.61E-06, 6.70E-06, 9.44E-06, 1.28E-05, 1.69E-05, 2.18E-05, 2.76E-05, 3.43E-05, 4.20E-05))
kact[2,:] = np.asarray((1.15E-06, 1.87E-06, 2.95E-06, 4.20E-06, 5.89E-06, 7.98E-06, 1.05E-05, 1.36E-05, 1.72E-05, 2.09E-05, 2.58E-05))
kact[3,:] = np.asarray((7.65E-07, 1.25E-06, 1.92E-06, 2.80E-06, 3.93E-06, 5.33E-06, 7.03E-06, 9.06E-06, 1.15E-05, 1.42E-05, 1.72E-05))
kact[4,:] = np.asarray((5.41E-07, 8.85E-07, 1.36E-06, 1.98E-06, 2.78E-06, 3.77E-06, 4.97E-06, 6.41E-06, 8.08E-06, 1.00E-05, 1.23E-05))

In [2]:
#Empirical parameters (2,4,6leg)
k001 = [3.66E-06, 1.84E-06, 1.15E-06]
kact = np.zeros((3, 11))
kact[0,:] = np.asarray((3.66E-06, 5.99E-06, 9.19E-06, 1.34E-05, 1.89E-05, 2.56E-05, 3.38E-05, 4.36E-05, 5.52E-05, 6.86E-05, 8.41E-05))
kact[1,:] = np.asarray((1.84E-06, 3.00E-06, 4.61E-06, 6.70E-06, 9.44E-06, 1.28E-05, 1.69E-05, 2.18E-05, 2.76E-05, 3.43E-05, 4.20E-05))
kact[2,:] = np.asarray((1.15E-06, 1.87E-06, 2.95E-06, 4.20E-06, 5.89E-06, 7.98E-06, 1.05E-05, 1.36E-05, 1.72E-05, 2.09E-05, 2.58E-05))

In [3]:
#Common parameters
k001 = np.multiply(k001,1e4)
kact = np.multiply(kact,1e4)
width = (0.01, 0.012, 0.014, 0.016, 0.018, 0.02, 0.022, 0.024, 0.026, 0.028)

In [4]:
p = xp.problem()
c = xp.vars(7, name="constants", vartype=xp.continuous)
pred = xp.vars(5, len(width), name="pred", vartype=xp.continuous)
p.addVariable(c)

In [5]:
p.addConstraint(c[0] + c[1]*0.01 + c[2]*0.01**2 + c[3]*0.01**3 + c[4]*0.01**4 == 1)

In [6]:
for i in range(3):
    for count,w in enumerate(width):
        pred[i,count] = (k001[i] * (c[0] + c[1]*w + c[2]*w**2 + c[3]*w**3 + c[4]*w**4))

sum_of_squares = (xp.Sum((kact[i,j] - pred[i,j])**2 for j in range(len(width)) for i in range(3) ))

p.setObjective(sum_of_squares, xp.minimize)

In [7]:
# p.controls.xslp_convergenceops = 6175 #6175=(bits 0-4, 11, 12), default=(bits 0-9, 11, 12)
p.solve()

FICO Xpress v8.10.1, Hyper, solve started 11:57:51, Jan 30, 2021
Heap usage: 331KB (peak 331KB, 521KB system)
Minimizing QP noname
Original problem has:
         1 rows            7 cols            5 elements
        25 qobjelem
Presolved problem has:
         1 rows            5 cols            5 elements
        25 qobjelem
Presolve finished in 0 seconds
Heap usage: 333KB (peak 343KB, 522KB system)

Coefficient range                    original                 solved        
  Coefficients   [min,max] : [ 1.00e-08,  1.00e+00] / [ 5.24e-01,  1.00e+00]
  RHS and bounds [min,max] : [ 1.00e+00,  1.00e+00] / [ 1.00e+00,  1.00e+00]
  Objective      [min,max] : [ 9.60e-08,  2.74e-01] / [ 2.74e-01,  6.44e+00]
  Quadratic      [min,max] : [ 2.88e-15,  3.62e-02] / [ 3.62e-02,  1.30e+01]
Autoscaling applied standard scaling

Barrier cache sizes : L1=16K L2=8192K
Using AVX support
Cores per CPU (CORESPERCPU): 8
Barrier starts after 0 seconds, using up to 8 threads, 4 cores
Matrix ordering - Dens

In [8]:
print (p.getObjVal())
print (p.getSolution(c[0]))
print (p.getSolution(c[1]))
print (p.getSolution(c[2]))
print (p.getSolution(c[3]))
print (p.getSolution(c[4]))
print (p.getSolution(c[5]))
print (p.getSolution(c[6]))

4.9966105932996996e-05
0.07800324980076745
0.7894153520727603
1077.7362434790814
804995.9434599403
133302.8870656524
0.0
0.0


In [9]:
for i in range(len(width)):
    print (p.getSolution(pred[i]))
    print (kact_real[i])

InterfaceError: xpress.evaluate() requires one or more Python objects;
NumPy arrays must be of dtype object.